In [1]:
TF_ENABLE_ONEDNN_OPTS=0

In [2]:
import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline, AutoTokenizer

2026-05-04 11:05:08.172270: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 11:05:08.184665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777892708.199849     160 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777892708.204863     160 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777892708.217304     160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

=================================================
LOAD MODEL
=================================================

Use Hugging Face's transformers library to load DistilGPT-2 — a lighter, faster version of GPT-2 (~80 MB). The pipeline wrapper abstracts away the tokenization and inference steps, we can just pass in a string and get text back

In [3]:
generator = pipeline("text-generation", model="distilgpt2")

print("Model loaded!\n")

Model loaded!



The show() Helper
A simple pretty-printer that formats the prompt and its generated outputs cleanly. Nothing model-related — just presentation logic.

In [4]:
# ════════════════════════════════════════════════════════════
#  HELPER — pretty-print results
# ════════════════════════════════════════════════════════════
def show(title: str, prompt: str, results: list, notes: str = ""):
    print("─" * 62)
    print(f"  {title}")
    if notes:
        print(f"  ℹ️  {notes}")
    print("─" * 62)
    print(f"  Prompt : \"{prompt}\"")
    for i, r in enumerate(results, 1):
        label = f"  Seq {i}  : " if len(results) > 1 else "  Output : "
        print(f"{label}{r['generated_text']}")
    print()

Test 1 — Baseline
A simple generation with default settings. Establishes what the model produces without tuning anything.

In [5]:
show(
    title  = "TEST 1 │ Baseline Prompt",
    prompt = "AI is transforming industries by",
    results= generator(
        "AI is transforming industries by",
        max_length=40,
        num_return_sequences=1
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 1 │ Baseline Prompt
──────────────────────────────────────────────────────────────
  Prompt : "AI is transforming industries by"
  Output : AI is transforming industries by design, manufacturing, education, health, and environmental sectors to create an environment in which only one part of one species is a member of the population, by creating a natural habitat



Test 2 — Different Topic
Same parameters as Test 1, but a different prompt. Shows that the model's output direction is entirely steered by the input text.

In [6]:
# ════════════════════════════════════════════════════════════
#  TEST 2 · Different topic — observe domain shift in output
# ════════════════════════════════════════════════════════════
show(
    title  = "TEST 2 │ Different Topic Prompt",
    prompt = "The future of medicine depends on",
    results= generator(
        "The future of medicine depends on",
        max_length=40,
        num_return_sequences=1
    ),
    notes  = "Same params, different topic → output changes direction"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 2 │ Different Topic Prompt
  ℹ️  Same params, different topic → output changes direction
──────────────────────────────────────────────────────────────
  Prompt : "The future of medicine depends on"
  Output : The future of medicine depends on a variety of factors. The most difficult part about these matters is the nature of medicine. For one, the current age is probably the oldest of the 20th Century,



Test 3 — Varying max_length
Runs the same prompt twice with max_length=20 and max_length=60. Demonstrates that max_length controls the total token count (prompt + output), so longer values allow the model to elaborate further.

In [7]:
# ════════════════════════════════════════════════════════════
#  TEST 3 · Vary max_length → see how output truncates / grows
# ════════════════════════════════════════════════════════════
prompt3 = "Space exploration will lead to"
print("─" * 62)
print("  TEST 3 │ Varying max_length (20 vs 60 tokens)")
print("  ℹ️  Watch how the story grows as we allow more tokens")
print("─" * 62)
print(f"  Prompt : \"{prompt3}\"")

r_short = generator(prompt3, max_length=20,  num_return_sequences=1)
r_long  = generator(prompt3, max_length=60,  num_return_sequences=1)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 3 │ Varying max_length (20 vs 60 tokens)
  ℹ️  Watch how the story grows as we allow more tokens
──────────────────────────────────────────────────────────────
  Prompt : "Space exploration will lead to"


In [8]:
print(f"  max=20  : {r_short[0]['generated_text']}")

  max=20  : Space exploration will lead to an eventual return of "the best that ever existed".


In [9]:
print(f"  max=60  : {r_long[0]['generated_text']}")

  max=60  : Space exploration will lead to significant investments in the space program," said NASA Administrator Jens Stoltenberg in a statement. "This is the first step toward a full-size mission, with the goal of establishing more affordable and cost-effective, science-driven space vehicles and instruments to replace aging


Test 4 — Multiple Sequences
Uses num_return_sequences=3 with do_sample=True and temperature=0.9. This generates three different completions from the same prompt in a single call, showing the natural diversity from sampling.

In [18]:
# ════════════════════════════════════════════════════════════
#  TEST 4 · Multiple sequences — diversity in one call
# ════════════════════════════════════════════════════════════
show(
    title  = "TEST 4 │ Multiple Sequences (num_return_sequences=3)",
    prompt = "Robots will soon be able to",
    results= generator(
        "Robots will soon be able to",
        max_length=45,
        num_return_sequences=3,
        do_sample=True,
        temperature=0.9
    ),
    notes  = "Three different continuations from the same prompt"
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 4 │ Multiple Sequences (num_return_sequences=3)
  ℹ️  Three different continuations from the same prompt
──────────────────────────────────────────────────────────────
  Prompt : "Robots will soon be able to"
  Seq 1  : Robots will soon be able to make more of a virtual reality experience. There's even a game for PC, but what's better for you, like this:


This is not a new concept. Instead,
  Seq 2  : Robots will soon be able to do so, with the addition of these tools to the development of robot tools.






















  Seq 3  : Robots will soon be able to access their data in-depth and user-friendly ways. For instance, it could be used to easily access a user›s location, for example, in a Google Maps dashboard.



Test 5 — Temperature Effect
Compares temperature=0.3 (low) vs temperature=1.5 (high). Temperature scales the probability distribution over the next token:

Low temp → model plays it safe, picks high-probability (repetitive) words
High temp → probabilities are flattened, model takes wilder guesses

In [11]:

# ════════════════════════════════════════════════════════════
#  TEST 5 · Temperature — focused (0.3) vs creative (1.5)
# ════════════════════════════════════════════════════════════
prompt5 = "Deep learning models can"
print("─" * 62)
print("  TEST 5 │ Temperature Effect (do_sample=True)")
print("  ℹ️  Low temp = repetitive/safe. High temp = wild/creative.")
print("─" * 62)
print(f"  Prompt : \"{prompt5}\"")

r_low  = generator(prompt5, max_length=40, num_return_sequences=1,
                   do_sample=True, temperature=0.3)
r_high = generator(prompt5, max_length=40, num_return_sequences=1,
                   do_sample=True, temperature=1.5)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 5 │ Temperature Effect (do_sample=True)
  ℹ️  Low temp = repetitive/safe. High temp = wild/creative.
──────────────────────────────────────────────────────────────
  Prompt : "Deep learning models can"


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [12]:

print(f"  temp=0.3: {r_low[0]['generated_text']}")


  temp=0.3: Deep learning models can be used to create a new learning model.





























In [13]:
print(f"  temp=1.5: {r_high[0]['generated_text']}")

  temp=1.5: Deep learning models can tell, say, how people act with emotion as long as their emotion plays an integrated role. To test what․ they use this type of emotion for emotion, one simple task


Test 6 — Greedy vs Sampling

do_sample=False → Greedy decoding: always picks the single most likely next token. Fully deterministic — same output every run.
do_sample=True → Random sampling: picks from the probability distribution, so output varies each run.

In [14]:
# ════════════════════════════════════════════════════════════
#  TEST 6 · Greedy vs Sampling (do_sample flag)
# ════════════════════════════════════════════════════════════
prompt6 = "The best way to learn programming is"
print("─" * 62)
print("  TEST 6 │ Greedy Decoding vs Random Sampling")
print("  ℹ️  Greedy always picks the highest-prob token → deterministic")
print("─" * 62)
print(f"  Prompt : \"{prompt6}\"")

r_greedy  = generator(prompt6, max_length=40, do_sample=False)   # greedy
r_sampled = generator(prompt6, max_length=40, do_sample=True, temperature=1.0)

print(f"  greedy  : {r_greedy[0]['generated_text']}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  TEST 6 │ Greedy Decoding vs Random Sampling
  ℹ️  Greedy always picks the highest-prob token → deterministic
──────────────────────────────────────────────────────────────
  Prompt : "The best way to learn programming is"


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  greedy  : The best way to learn programming is to learn from the experience of the programmer.


























In [15]:
print(f"  sampled : {r_sampled[0]['generated_text']}")

  sampled : The best way to learn programming is using the best techniques, with all sorts of programming languages, so that you're already familiar with how your students can learn. With C#, you can learn programming


Test 7 — Tokenisation Deep-Dive
Uses the raw AutoTokenizer to show how GPT-2 splits text into subword tokens using Byte-Pair Encoding (BPE). Key insight: word count ≠ token count. For example, "Unbelievably" might be split into ["Un", "believ", "ably"] — three tokens from one word.

In [16]:
# ════════════════════════════════════════════════════════════
#  TEST 7 · Tokenisation deep-dive
# ════════════════════════════════════════════════════════════
print("─" * 62)
print("  TEST 7 │ Tokenisation Peek")
print("  ℹ️  Words ≠ tokens. GPT-2 uses Byte-Pair Encoding (BPE).")
print("─" * 62)

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

sentences = [
    "AI is transforming industries by automating tasks",
    "Unbelievably, transformers revolutionised NLP overnight!",
]

for sentence in sentences:
    token_ids = tokenizer.encode(sentence)
    tokens    = [tokenizer.decode([t]) for t in token_ids]
    print(f"\n  Sentence  : {sentence}")
    print(f"  Token IDs : {token_ids}")
    print(f"  Tokens    : {tokens}")
    print(f"  Stats     : {len(sentence.split())} words → {len(token_ids)} tokens")

print()

──────────────────────────────────────────────────────────────
  TEST 7 │ Tokenisation Peek
  ℹ️  Words ≠ tokens. GPT-2 uses Byte-Pair Encoding (BPE).
──────────────────────────────────────────────────────────────

  Sentence  : AI is transforming industries by automating tasks
  Token IDs : [20185, 318, 25449, 11798, 416, 3557, 803, 8861]
  Tokens    : ['AI', ' is', ' transforming', ' industries', ' by', ' autom', 'ating', ' tasks']
  Stats     : 7 words → 8 tokens

  Sentence  : Unbelievably, transformers revolutionised NLP overnight!
  Token IDs : [3118, 6667, 11203, 1346, 11, 6121, 364, 5854, 1417, 399, 19930, 13417, 0]
  Tokens    : ['Un', 'bel', 'iev', 'ably', ',', ' transform', 'ers', ' revolution', 'ised', ' N', 'LP', ' overnight', '!']
  Stats     : 5 words → 13 tokens



Summary Table at the End
Prints a quick-reference cheat sheet mapping each parameter to its observed effect, making it easy to remember the key takeaways from all 7 tests.

In [17]:
# ════════════════════════════════════════════════════════════
#  SUMMARY TABLE
# ════════════════════════════════════════════════════════════
print("=" * 62)
print("  WHAT TO OBSERVE — Key Takeaways")
print("=" * 62)
summary = [
    ("max_length ↑",         "Longer output; model may drift off-topic"),
    ("num_return_sequences↑","Multiple diverse completions in one call"),
    ("temperature ↓ (0.1)",  "Safe, repetitive, predictable text"),
    ("temperature ↑ (1.5)",  "Creative, sometimes incoherent text"),
    ("do_sample=False",      "Greedy — same output every run"),
    ("do_sample=True",       "Stochastic — different each time"),
    ("Tokenisation",         "Subword tokens, not whole words (BPE)"),
]
print(f"  {'Parameter':<28} {'Effect'}")
print("  " + "-" * 56)
for param, effect in summary:
    print(f"  {param:<28} {effect}")
print("=" * 62)
print("  ✅  All tests complete!")
print("=" * 62)

  WHAT TO OBSERVE — Key Takeaways
  Parameter                    Effect
  --------------------------------------------------------
  max_length ↑                 Longer output; model may drift off-topic
  num_return_sequences↑        Multiple diverse completions in one call
  temperature ↓ (0.1)          Safe, repetitive, predictable text
  temperature ↑ (1.5)          Creative, sometimes incoherent text
  do_sample=False              Greedy — same output every run
  do_sample=True               Stochastic — different each time
  Tokenisation                 Subword tokens, not whole words (BPE)
  ✅  All tests complete!


 This code is a parameter sensitivity study — it isolates one variable at a time so you can clearly see how each knob changes the model's behavior.

In [0]:
 Section A - LLM Foundations & Hugging Face - Completed

1. The word_count() Helper
A simple utility that splits the generated text by spaces and counts the resulting list. Used to check whether the summarisation output actually stays within the ≤ 30 word target.

In [19]:
# ────────────────────────────────────────────────────────────
#  HELPER — word counter to check output length
# ────────────────────────────────────────────────────────────
def word_count(text: str) -> int:
    return len(text.split())

2. The show_b() Helper
    This is the Section B version of the original show() function. The key difference is it accepts lists of prompts and results and uses zip() to pair each prompt with its corresponding output, printing them together for easy side-by-side comparison. The word count is shown alongside each output

In [20]:
def show_b(title: str, prompts: list, results: list, notes: str = ""):
    """Display side-by-side prompt comparisons for Section B."""
    print("─" * 62)
    print(f"  {title}")
    if notes:
        print(f"  ℹ️  {notes}")
    print("─" * 62)
    for i, (prompt, result) in enumerate(zip(prompts, results), 1):
        output = result[0]['generated_text']
        wc     = word_count(output)
        print(f"\n  Prompt v{i} : \"{prompt}\"")
        print(f"  Output    : {output}")
        print(f"  Word count: {wc} words")
    print()

Task 1 — Summarisation
Three prompts are designed with increasing levels of instruction clarity:

        v1 - NothingBaseline           - see what the model does freely
        v2 - In one sentence           - Gives a format constraint
        v3 - Source text + word limit  - Anchors the model to specific content


All three use do_sample=False (greedy decoding) so any output differences come purely from the prompt wording, not randomness.

In [30]:

# ════════════════════════════════════════════════════════════
#  TASK 1 · SUMMARISATION
#  Goal : Keep output ≤ 30 words.
#  Strategy : Compare vague vs specific instruction phrasing.
# ════════════════════════════════════════════════════════════
print("━" * 62)
print("  TASK 1 │ SUMMARISATION  (target: ≤ 30 words output)")
print("━" * 62)

# Prompt v1 — Vague: no length or style instruction
sum_prompt_v1 = "Summarize AI"

# Prompt v2 — Clear: specifies brevity and a sentence cap
sum_prompt_v2 = "In one sentence, briefly summarize Vector embedding"

# Prompt v3 — Structured: gives context + explicit instruction
sum_prompt_v3 = (
    "Food waste carries a significant environmental toll by squandering resources like water, fertilizer, and fuel used in production, and if treated as a country, it would be the world's third-largest emitter of greenhouse gases. "
    "Summarize this in 20 words or fewer:"
)

sum_prompts = [sum_prompt_v1, sum_prompt_v2, sum_prompt_v3]
sum_results = [
    generator(p, max_length=30, num_return_sequences=1, do_sample=False)
    for p in sum_prompts
]

show_b(
    title   = "SUMMARISATION — Prompt Comparison",
    prompts = sum_prompts,
    results = sum_results,
    notes   = "v1=vague | v2=adds brevity cue | v3=context + length constraint"
)

print("Observation:")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 1 │ SUMMARISATION  (target: ≤ 30 words output)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  SUMMARISATION — Prompt Comparison
  ℹ️  v1=vague | v2=adds brevity cue | v3=context + length constraint
──────────────────────────────────────────────────────────────

  Prompt v1 : "Summarize AI"
  Output    : Summarize AI.

























  Word count: 2 words

  Prompt v2 : "In one sentence, briefly summarize Vector embedding"
  Output    : In one sentence, briefly summarize Vector embedding.



The first sentence is a simple example of the concept of embedding.
The
  Word count: 20 words

  Prompt v3 : "Food waste carries a significant environmental toll by squandering resources like water, fertilizer, and fuel used in production, and if treated as a country, it would be the world's third-largest emitter of greenhouse gases. Summarize this in 20 words or fewer:"
  Output    : Food waste carries a significant environmental toll by squandering resources like water, fertilizer, and fuel used in production, and i

Task 2 — Q&A
Three different prompt strategies for factual answering:

v1 — A plain question. DistilGPT-2 often just rephrases or echoes the question back because it wasn't trained as a Q&A model.
v2 — Adds a role ("As an AI expert") and a format instruction ("one sentence"). This nudges the model toward a more informative, confident tone.
v3 — A sentence-completion style. Instead of asking a question, you give the model the beginning of the answer. This is the most effective trick for small models — it removes ambiguity about what kind of response is expected.

In [32]:

# ════════════════════════════════════════════════════════════
#  TASK 2 · QUESTION & ANSWER (Q&A)
#  Goal : Elicit a factual, direct answer.
#  Strategy : Compare open-ended vs answer-framing prompts.
# ════════════════════════════════════════════════════════════
print("━" * 62)
print("  TASK 2 │ Q&A  (factual question answering)")
print("━" * 62)

# Prompt v1 — Plain question only
qa_prompt_v1 = "What is Mathematics?"

# Prompt v2 — Role + question format
qa_prompt_v2 = "As an expert mathematian, answer in one sentence: What is Maths?"

# Prompt v3 — Fill-in-the-blank style (strong answer framing)
qa_prompt_v3 = "Calculas is a branch of Maths that"

qa_prompts = [qa_prompt_v1, qa_prompt_v2, qa_prompt_v3]
qa_results = [
    generator(p, max_length=55, num_return_sequences=1, do_sample=False)
    for p in qa_prompts
]

show_b(
    title   = "Q&A — Prompt Comparison",
    prompts = qa_prompts,
    results = qa_results,
    notes   = "v1=plain | v2=role+format | v3=sentence-completion framing"
)

print("  📝 Observation:")
print("     v1 (plain)       → Model may repeat or rephrase the question.")
print("     v2 (role+format) → 'Expert' framing encourages factual language.")
print("     v3 (completion)  → Sentence-start forces the model straight to answer.")
print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 2 │ Q&A  (factual question answering)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  Q&A — Prompt Comparison
  ℹ️  v1=plain | v2=role+format | v3=sentence-completion framing
──────────────────────────────────────────────────────────────

  Prompt v1 : "What is Mathematics?"
  Output    : What is Mathematics?



















































  Word count: 3 words

  Prompt v2 : "As an expert mathematian, answer in one sentence: What is Maths?"
  Output    : As an expert mathematian, answer in one sentence: What is Maths?



The answer is that the answer is that the answer is that the answer is that the answer is that the answer is that the answer is that the answer is that the answer is
  Word count: 46 words

  Prompt v3 : "Calculas is a branch of Maths that"
  Output    : Calculas is a branch of Maths that is used to calculate the number of times a given number of times a given number of times a given number of times a given number of times a given number of times a given number of time

5. Task 3 — Creative Text (Poem)
Each version adds more creative scaffolding:

v1 — Bare request. Small models like DistilGPT-2 tend to produce prose rather than actual verse because the instruction is too open-ended.
v2 — Specifies "4-line" and "rhyming". These structural keywords push the model toward verse-shaped output.
v3 — Provides the first line of the poem. This is the strongest technique — the model simply continues from where you left off, inheriting the rhythm, theme, and style you set.
Higher temperature is intentional here — creativity tasks benefit from some randomness rather than always picking the safest next word.

In [35]:

# ════════════════════════════════════════════════════════════
#  TASK 3 · CREATIVE TEXT GENERATION
#  Goal : Generate a 4-line poem on AI.
#  Strategy : Compare bare request vs structured creative prompts.
# ════════════════════════════════════════════════════════════
print("━" * 62)
print("  TASK 3 │ CREATIVE TEXT  (4-line poem on AI)")
print("━" * 62)

# Prompt v1 — Bare request
creative_prompt_v1 = "Write a poem about artificial intelligence."

# Prompt v2 — Specifies structure (4 lines) + mood
creative_prompt_v2 = "Write a 4-line rhyming poem about artificial intelligence and the future:"

# Prompt v3 — Gives the first line as a creative anchor
creative_prompt_v3 = (
    "Complete this 4-line poem about AI:\n"
    "In circuits and code, a new mind is born,\n"
)

creative_prompts = [creative_prompt_v1, creative_prompt_v2, creative_prompt_v3]
creative_results = [
    generator(
        p,
        max_length      = 80,
        num_return_sequences = 1,
        do_sample       = True,
        temperature     = 0.9      # higher temp for more creative output
    )
    for p in creative_prompts
]

show_b(
    title   = "CREATIVE TEXT — Prompt Comparison",
    prompts = creative_prompts,
    results = creative_results,
    notes   = "v1=bare | v2=structure+mood | v3=anchor line provided"
)

print("  📝 Observation:")
print("     v1 (bare)    → Output may be prose-like, not poem-shaped.")
print("     v2 (specific)→ 'Rhyming' cue pushes toward verse structure.")
print("     v3 (anchor)  → Giving line 1 strongly constrains style & theme.")
print()


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 3 │ CREATIVE TEXT  (4-line poem on AI)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


──────────────────────────────────────────────────────────────
  CREATIVE TEXT — Prompt Comparison
  ℹ️  v1=bare | v2=structure+mood | v3=anchor line provided
──────────────────────────────────────────────────────────────

  Prompt v1 : "Write a poem about artificial intelligence."
  Output    : Write a poem about artificial intelligence.



The goal of this project was to document the most powerful tools for analyzing the human brain through human brain-computer interaction (BDI). The work was described in the Proceedings of the National Academy of Sciences.
The goal of this project was to document the most powerful tools for analyzing the human brain through human brain-computer interaction (BDI).
  Word count: 63 words

  Prompt v2 : "Write a 4-line rhyming poem about artificial intelligence and the future:"
  Output    : Write a 4-line rhyming poem about artificial intelligence and the future: "I am a big believer in AI."
  Word count: 18 words

  Prompt v3 : "Complete this 4-line 

6. The Comparison Loop Pattern
All three tasks follow the same pattern:
A list comprehension runs the generator on every prompt and collects all results into one list. This keeps the code clean and avoids repeating the generator() call three separate times.

In [36]:

# ════════════════════════════════════════════════════════════
#  PROMPT COMPARISON SUMMARY TABLE
# ════════════════════════════════════════════════════════════
print("=" * 62)
print("  SECTION B — Prompt Engineering Summary")
print("=" * 62)
pe_summary = [
    ("Task",          "Prompt Style",      "Key Effect"),
    ("─" * 14,        "─" * 18,            "─" * 24),
    ("Summarisation", "Vague (v1)",         "Verbose, unfocused output"),
    ("",              "Clear (v2)",         "Shorter, more on-topic"),
    ("",              "Context+limit (v3)", "Best anchored summary"),
    ("Q&A",           "Plain (v1)",         "May echo the question"),
    ("",              "Role+format (v2)",   "More factual tone"),
    ("",              "Completion (v3)",    "Direct, concise answer"),
    ("Creative",      "Bare (v1)",          "Prose drift, no structure"),
    ("",              "Structured (v2)",    "More verse-like output"),
    ("",              "Anchor line (v3)",   "Strongest style control"),
]
for row in pe_summary:
    print(f"  {row[0]:<14} {row[1]:<20} {row[2]}")
print()

  SECTION B — Prompt Engineering Summary
  Task           Prompt Style         Key Effect
  ────────────── ──────────────────   ────────────────────────
  Summarisation  Vague (v1)           Verbose, unfocused output
                 Clear (v2)           Shorter, more on-topic
                 Context+limit (v3)   Best anchored summary
  Q&A            Plain (v1)           May echo the question
                 Role+format (v2)     More factual tone
                 Completion (v3)      Direct, concise answer
  Creative       Bare (v1)            Prose drift, no structure
                 Structured (v2)      More verse-like output
                 Anchor line (v3)     Strongest style control



Section B — Prompt Engineering Completed